# Hi! 👋

The following notebook is used to **fine tune experiments' clients**. If you are running the following notebook in Colab, the next cell will ask you for a GDrive URL and a github token. The token will be used to clone [DecentralizedLearning/dEngine](https://github.com/DecentralizedLearning/dEngine). If you don't own the repo, the token can be generated using Github web (`settings > Developer Settings > Personal Access Tokens`). The GDrive URL will be provided by who ran the experiments. 

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import gdown

try:  # Check if we are in Google colab
    import google.colab
    from google.colab import output
    output.enable_custom_widget_manager()
    
    !git clone --quiet https://github.com/DecentralizedLearning/dEngine ..
    !pip install --quiet git+https://github.com/DecentralizedLearning/notebooks.git
    
    EXPERIMENT_GDRIVE_URL = input('Experiment remote URL \t')
    LOGS_PATH = Path('/content/logs')
    if not os.path.exists(LOGS_PATH):
        gdown.download_folder(url=EXPERIMENT_GDRIVE_URL, output=str(LOGS_PATH))
        !unzip -q {LOGS_PATH}/\*.zip -d {LOGS_PATH}    
except:
    pass

<br><br>
<hr>

<br><br>

# Experiment Selection

The next cell will open a file selection widget.

Please select the directory containing the experiment logs (usually inside a `log/` folder).

**Note**: If you are using Google Drive, you may need to navigate one level up. Logs are typically downloaded to `/content/logs`, while the code runs in `/content/SaiSim`.

In [ ]:
from dnotebooks.widgets import MultiExperimentSelection

experiment_selection_widget, get_selected_experiments = MultiExperimentSelection(limit=1)
display(experiment_selection_widget)

<br><br>
<hr>
<br><br>

In [ ]:
from torch.utils.data import Subset
from dengine.analysis.experiment import Experiment

selected_experiment_path = get_selected_experiments()[0]
selected_experiment = Experiment(
    experiment_root_path=selected_experiment_path, 
    dataset_root_path=Path("./datasets/")
)

model = selected_experiment.training_engine.clients['0'].model
D_tr = Subset(selected_experiment.dataset_train, selected_experiment.training_engine.partitioning['1']['train'])
D_ts = selected_experiment.dataset_test

In [ ]:
%matplotlib inline

from copy import deepcopy
from pathlib import Path
import random

import torch
from torch.utils.data import DataLoader
import torch.nn as nn
from tqdm.notebook import tqdm
import numpy as np

import matplotlib.pyplot as plt
from dengine.utils.utils import get_output_in_production, seed_everything
from dengine.utils.confusion_matrix import confusion_matrix_from_model_output
from dengine.analysis.confusion_matrix import _normalize_confusion_matrix_rows

from dnotebooks.plots import confusion_matrix_heatmap

seed_everything(42)


# ............... ............... ............... #
# PERFORMANCE WITHOUT FINE-TUNING
# ............... ............... ............... #
Y_test_hat = get_output_in_production(model, D_ts, None, 128)
test_conf_matrix = confusion_matrix_from_model_output(Y_test_hat, D_ts, None, 10).squeeze()
n_test_conf_matrix = _normalize_confusion_matrix_rows(test_conf_matrix)

confusion_matrix_heatmap(n_test_conf_matrix, None)
title = f"Node-0 before finetuning\n({selected_experiment_path.name})"
plt.title(title)
Path("res").mkdir(exist_ok=True, parents=True)
plt.savefig(f"res/{title}.png", bbox_inches='tight', pad_inches=0)
plt.show()


# ............... ............... ............... #
# FINE-TUNING CLIENT MODEL ON D_TR
# ............... ............... ............... #
finetuned_model = deepcopy(model)
EPOCHS = 15

# Freeze all layers except the last
for param in finetuned_model.parameters():
    param.requires_grad = False
for param in list(finetuned_model.parameters())[-2:]:
    param.requires_grad = True

# Optimizer, loss, dataset
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, finetuned_model.parameters()), lr=0.02)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-9)
loader = DataLoader(D_tr, batch_size=32, shuffle=True)

# Training loop
import sys

best_model = finetuned_model
min_loss = sys.maxsize
finetuned_model.train()
for e in tqdm(range(EPOCHS), leave=False, desc="Epochs"):
    running_loss = 0
    for X, y in tqdm(loader, leave=False, desc="Batch"):
        optimizer.zero_grad()
        loss = criterion(finetuned_model(X), y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    epoch_loss = running_loss / len(loader)
    if epoch_loss < min_loss:
        best_model = deepcopy(finetuned_model)
        min_loss = epoch_loss
        best_epoch = e
        print(f"* Epoch {e+1} — Loss: {epoch_loss:.4f}")
    else:
        print(f"Epoch {e+1} — Loss: {epoch_loss:.4f}")
    scheduler.step()


# ............... ............... ............... #
# PERFORMANCE WITH FINE-TUNING
# ............... ............... ............... #
Y_test_hat = get_output_in_production(best_model, D_ts, None, 128)
test_conf_matrix = confusion_matrix_from_model_output(Y_test_hat, D_ts, None, 10).squeeze()

fig, _ = plt.subplots()
_ax = fig.axes[0]
confusion_matrix_heatmap(_normalize_confusion_matrix_rows(test_conf_matrix), None)
title = f"Node-0 after finetuning head on D_tr^49\n({selected_experiment_path.name}, best at e{best_epoch})"
plt.title(title)
plt.show()
plt.savefig(f"res/{title}.png", bbox_inches='tight', pad_inches=0)

<br><br>
<hr>

<br><br>